# Logic Gate Learning using Perceptron

A perceptron is the simplest neural network unit. It takes binary inputs and maps them to a binary output using a weighted sum followed by a threshold function.

Here we train a perceptron on different logic gates — AND, OR, NAND, NOR, and XOR. XOR is a special case since it cannot be learned by a single perceptron (not linearly separable).

In [2]:

gate_data = {
    "AND":  [(0,0,0), (0,1,0), (1,0,0), (1,1,1)],
    "OR":   [(0,0,0), (0,1,1), (1,0,1), (1,1,1)],
    "NAND": [(0,0,1), (0,1,1), (1,0,1), (1,1,0)],
    "NOR":  [(0,0,1), (0,1,0), (1,0,0), (1,1,0)],
    "XOR":  [(0,0,0), (0,1,1), (1,0,1), (1,1,0)]
}

print("Gate truth tables loaded:", list(gate_data.keys()))

Gate truth tables loaded: ['AND', 'OR', 'NAND', 'NOR', 'XOR']


## Step Function and Forward Pass

The perceptron computes a weighted sum of its inputs plus a bias term:

$$z = w_a \cdot a + w_b \cdot b + bias$$

This is then passed through a **step activation function** — output is 1 if $z \geq 0$, else 0.

In [3]:
def step_activate(val):
    return 1 if val >= 0 else 0

def forward(a, b, wa, wb, bias):

    net_input = (wa * a) + (wb * b) + bias

    output = step_activate(net_input)
    return output

## Perceptron Training

The perceptron learning rule updates weights when a prediction is wrong:

$$w \leftarrow w + \eta \cdot (y - \hat{y}) \cdot x$$

- $\eta$ is the learning rate (set to 0.1)
- Training stops early if all predictions are correct in an epoch

In [4]:
def run_training(samples, lr=0.1, max_epochs=10):

    wa, wb, bias = 0.0, 0.0, 0.0

    for ep in range(max_epochs):
        mistakes = 0

        for a, b, target in samples:
            pred = forward(a, b, wa, wb, bias)
            delta = target - pred


            if delta != 0:
                wa   += lr * delta * a
                wb   += lr * delta * b
                bias += lr * delta
                mistakes += abs(delta)


        if mistakes == 0:
            break

    return wa, wb, bias

## Training and Results

Train the perceptron on each gate and verify predictions against the truth table.

In [7]:
for name, samples in gate_data.items():
    w_a, w_b, b = run_training(samples)

    print(f"\n {name} Gate ")
    print(f"  Learned weights: wa={w_a:.2f}, wb={w_b:.2f}, bias={b:.2f}")

    all_correct = True
    for a, b_val, expected in samples:
        got = forward(a, b_val, w_a, w_b, b)
        result = "OK" if got == expected else "WRONG"
        if got != expected:
            all_correct = False
        print(f"  ({a}, {b_val}) -> expected={expected}, got={got}  [{result}]")

    if name == "XOR" and not all_correct:
        print("   XOR cannot be learned by a single perceptron (not linearly separable).")



 AND Gate 
  Learned weights: wa=0.20, wb=0.10, bias=-0.20
  (0, 0) -> expected=0, got=0  [OK]
  (0, 1) -> expected=0, got=0  [OK]
  (1, 0) -> expected=0, got=0  [OK]
  (1, 1) -> expected=1, got=1  [OK]

 OR Gate 
  Learned weights: wa=0.10, wb=0.10, bias=-0.10
  (0, 0) -> expected=0, got=0  [OK]
  (0, 1) -> expected=1, got=1  [OK]
  (1, 0) -> expected=1, got=1  [OK]
  (1, 1) -> expected=1, got=1  [OK]

 NAND Gate 
  Learned weights: wa=-0.20, wb=-0.10, bias=0.20
  (0, 0) -> expected=1, got=1  [OK]
  (0, 1) -> expected=1, got=1  [OK]
  (1, 0) -> expected=1, got=1  [OK]
  (1, 1) -> expected=0, got=0  [OK]

 NOR Gate 
  Learned weights: wa=-0.10, wb=-0.10, bias=0.00
  (0, 0) -> expected=1, got=1  [OK]
  (0, 1) -> expected=0, got=0  [OK]
  (1, 0) -> expected=0, got=0  [OK]
  (1, 1) -> expected=0, got=0  [OK]

 XOR Gate 
  Learned weights: wa=-0.10, wb=0.00, bias=0.00
  (0, 0) -> expected=0, got=1  [WRONG]
  (0, 1) -> expected=1, got=1  [OK]
  (1, 0) -> expected=1, got=0  [WRONG]
  (1, 1)